# Lab 03: AI Agent with Strands SDK on Amazon Bedrock

In this lab we build an **AI operations agent** that manages TLS/SSL certificate lifecycle using the [Strands Agents SDK](https://github.com/strands-agents/sdk-python). The agent runs directly in this notebook kernel and calls AWS Lambda functions as tools.

**Architecture:**
- Strands Agent (local) → Amazon Bedrock (Claude) → Lambda tool calls → certificate operations

## 1. Load Configuration

In [ ]:
import json, pathlib, boto3

config = json.loads(pathlib.Path('/tmp/certagent_config.json').read_text())
globals().update(config)

REPO_DIR = pathlib.Path(REPO_DIR)
lm = boto3.client('lambda', region_name=AWS_REGION)

PRIORITY_EMOJI = {'EXPIRED': '\ud83d\udc80', 'CRITICAL': '\ud83d\udd34', 'HIGH': '\ud83d\udfe0', 'MEDIUM': '\ud83d\udfe1', 'LOW': '\ud83d\udfe2'}

def invoke(fn, payload):
    r = lm.invoke(FunctionName=fn, InvocationType='RequestResponse',
                  Payload=json.dumps(payload))
    raw = json.loads(r['Payload'].read())
    if 'FunctionError' in r:
        raise RuntimeError(raw.get('errorMessage', raw))
    return raw.get('body', raw)

print(f'Region  : {AWS_REGION}')
print(f'Lambda  : {LAMBDA_SCAN}')
print('\u2705 Environment ready')

## 2. Install Strands Agents SDK

In [ ]:
!pip install -q strands-agents strands-agents-bedrock

## 3. Define Tool Functions

Each tool wraps a Lambda function invocation. The agent will choose which tool(s) to call based on the user's request.

In [ ]:
from strands import tool


@tool
def scan_certificates(days_threshold: int = 30, use_mock: bool = True) -> str:
    """Scan for certificates that are expiring within a given number of days.

    Args:
        days_threshold: Number of days to look ahead for expiring certificates. Defaults to 30.
        use_mock: Whether to use mock data for the workshop. Defaults to True.

    Returns:
        JSON string with scan results including expiring certificates and their priorities.
    """
    payload = {
        'days_threshold': days_threshold,
        'use_mock': use_mock
    }
    result = invoke(LAMBDA_SCAN, payload)
    return json.dumps(result, indent=2) if isinstance(result, (dict, list)) else str(result)


@tool
def renew_certificate(domain_name: str, use_mock: bool = True) -> str:
    """Submit a renewal request for a specific certificate by domain name.

    Args:
        domain_name: The fully qualified domain name of the certificate to renew.
        use_mock: Whether to use mock mode for the workshop. Defaults to True.

    Returns:
        JSON string with the renewal request result including status and tracking info.
    """
    payload = {
        'domain_name': domain_name,
        'use_mock': use_mock
    }
    result = invoke(LAMBDA_RENEW, payload)
    return json.dumps(result, indent=2) if isinstance(result, (dict, list)) else str(result)


@tool
def check_status(domain_name: str, use_mock: bool = True) -> str:
    """Check the current status of a certificate or a pending renewal request.

    Args:
        domain_name: The fully qualified domain name to check status for.
        use_mock: Whether to use mock mode for the workshop. Defaults to True.

    Returns:
        JSON string with the certificate status details.
    """
    payload = {
        'domain_name': domain_name,
        'use_mock': use_mock
    }
    result = invoke(LAMBDA_STATUS, payload)
    return json.dumps(result, indent=2) if isinstance(result, (dict, list)) else str(result)


@tool
def list_inventory(use_mock: bool = True) -> str:
    """List the full certificate inventory with status information.

    Args:
        use_mock: Whether to use mock mode for the workshop. Defaults to True.

    Returns:
        JSON string with complete certificate inventory grouped by status.
    """
    payload = {
        'use_mock': use_mock
    }
    result = invoke(LAMBDA_INVENTORY, payload)
    return json.dumps(result, indent=2) if isinstance(result, (dict, list)) else str(result)


print('\u2705 Tools defined: scan_certificates, renew_certificate, check_status, list_inventory')

## 4. Create the Strands Agent

We instantiate the agent with the Bedrock model and register our 4 tools.

In [ ]:
from strands import Agent
from strands.models.bedrock import BedrockModel

SYSTEM_PROMPT = """You are CertAgent, an AI operations agent that manages TLS/SSL certificate lifecycle.
Capabilities: SCAN (find expiring certs), RENEW (submit renewal), CHECK STATUS, INVENTORY.
Rules:
- Always use mock mode (use_mock=true) in this workshop
- Report priority with emoji: \ud83d\udc80 Expired, \ud83d\udd34 Critical, \ud83d\udfe0 High, \ud83d\udfe1 Medium, \ud83d\udfe2 Low
- Be concise and operational
- After any action, state what was done and what happens next"""

bedrock_model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-5-20250514-v1:0",
    region_name=AWS_REGION
)

agent = Agent(
    model=bedrock_model,
    system_prompt=SYSTEM_PROMPT,
    tools=[scan_certificates, renew_certificate, check_status, list_inventory]
)

print('\u2705 CertAgent ready (Strands SDK + Bedrock Claude)')

## 5. Ask Helper

A simple helper to send a message to the agent and print the response.

In [ ]:
def ask(message: str):
    """Send a message to CertAgent and print the response."""
    print(f'\n\ud83d\udcac User: {message}\n')
    print('-' * 60)
    response = agent(message)
    print(f'\n\ud83e\udd16 CertAgent:\n{response}')
    print('-' * 60)

## 6. Conversation Examples

Let's interact with CertAgent through a series of operational tasks.

### 6.1 Scan for Expiring Certificates

In [ ]:
ask("What certs are expiring soon? Prioritized summary.")

### 6.2 Renew a Certificate

In [ ]:
ask("Renew api.example.com using mock mode")

### 6.3 Check Inventory

In [ ]:
ask("Show full inventory grouped by status")

### 6.4 Multi-Step: Scan and Renew Critical

In [ ]:
ask("Scan for certs expiring in 7 days, then renew any CRITICAL ones")

---

## \u2705 Lab 03 Complete!

**What we accomplished:**
- Built an AI agent using the **Strands Agents SDK** running locally in the notebook
- Connected it to **Amazon Bedrock** (Claude) for reasoning
- Registered **4 Lambda-backed tools** for certificate operations
- Demonstrated single-step and multi-step agentic workflows

**Key takeaways:**
- No classic Bedrock Agent API needed \u2014 the agent runs in-process
- Tools are plain Python functions decorated with `@tool`
- The agent autonomously decides which tools to call and in what order
- Multi-step reasoning (scan \u2192 filter \u2192 renew) happens automatically